# Δημιουργία του test set (2.000 προτάσεις)

Επιλέγουμε 2.000 ζεύγη EL-EN από τα 300.000 της προήγουμενης εργασίας, εξασφαλίζοντας να μην υπάρχουν near-duplicates, δηλαδή σχεδόν ίδιες προτάσεις. Τα πανομοιότυπα ζεύγη (exact duplicates) έχουν ήδη αφαιρεθεί στη διαδικασία καθαρισμού του Assignment 1. 
Ένα παράδειγμα near-duplicates αποτελεί το ζεύγος 
- «Which airlines provide the cheapest tickets from Zurich to Stuttgart» και
- «Which airlines provide the cheapest tickets from Houston to Stuttgart».

In [1]:
import os

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

with open("work/greek_sentences.txt", encoding="utf-8") as f:
    greek = f.read().splitlines()
with open("work/english_sentences.txt", encoding="utf-8") as f:
    english = f.read().splitlines()

corpus = pd.DataFrame({"greek": greek, "english": english})
len(corpus)

300000

## Ομοιότητα προτάσεων

Σπάμε κάθε πρόταση σε n-grams 3 έως 5 χαρακτήρων και δίνουμε μεγαλύτερο βάρος στα πιο σπάνια n-grams του corpus (TF-IDF). Στη συνέχεια υπολογίζουμε το cosine similarity (1 για ίδιες προτάσεις, 0 αν δεν έχουν κανένα κοινό n-gram) **ξεχωριστά για τα ελληνικά και τα αγγλικά**. Θεωρούμε δύο ζεύγη near-duplicates όταν έχουν ομοιότητα τουλάχιστον 0,8 σε μία από τις δύο γλώσσες (π.χ. οι προτάσεις Zurich/Houston έχουν ομοιότητα 0,84).

In [2]:
SIMILARITY_THRESHOLD = 0.8

tfidf = {
    lang: TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5)).fit_transform(corpus[lang])
    for lang in ["greek", "english"]
}

## Επιλογή των 2.000 προτάσεων

Για να επιλέξουμε από το αρχικό corpus των 300000 προτάσεων τελικά 2000 εφαρμόζουμε την εξής λογική: ανακατεύουμε τυχαία τα ζεύγη (με seed 42 για επαναληψιμότητα) και τα διατρέχουμε ένα-ένα. Κρατάμε κάθε ζεύγος αν δεν είναι near-duplicate με κάποιο από όσα έχουμε ήδη κρατήσει, μέχρι να συμπληρώσουμε 2.000. Με αυτό τον τρόπο αποφεύγουμε να υπολογίσουμε cosine similarity για όλα τα πιθανά ζεύγη προτάσεων του corpus.

In [3]:
TEST_SIZE = 2000

shuffled_corpus = corpus.sample(frac=1, random_state=42).index
test_rows = []
rejected = 0

for i in shuffled_corpus:
    if test_rows:
        sim_el = cosine_similarity(tfidf["greek"][i], tfidf["greek"][test_rows]).max()
        sim_en = cosine_similarity(tfidf["english"][i], tfidf["english"][test_rows]).max()
        if sim_el >= SIMILARITY_THRESHOLD or sim_en >= SIMILARITY_THRESHOLD:
            rejected += 1
            continue
    test_rows.append(i)
    if len(test_rows) == TEST_SIZE:
        break

print(f"Ζεύγη που απορρίφθηκαν: {rejected}")
test_set = corpus.loc[test_rows]

Ζεύγη που απορρίφθηκαν: 4


## Αφαίρεση από το υπόλοιπο corpus

Αφαιρούμε από το υπόλοιπο corpus όσα ζεύγη είναι near-duplicates με κάποιο ζεύγος του test set, ώστε να μην υπάρχουν στα δεδομένα εκπαίδευσης προτάσεις ίδιες ή σχεδόν ίδιες με το test set.

In [4]:
near_duplicates = set()

for i in test_rows:
    sim_el = cosine_similarity(tfidf["greek"][i], tfidf["greek"])[0]
    sim_en = cosine_similarity(tfidf["english"][i], tfidf["english"])[0]
    matches = np.where((sim_el >= SIMILARITY_THRESHOLD) | (sim_en >= SIMILARITY_THRESHOLD))[0]
    near_duplicates.update(matches)

remainder = corpus.drop(index=near_duplicates)
print(f"Αφαιρέθηκαν από το υπόλοιπο corpus: {len(near_duplicates) - len(test_set)}")
print(f"Υπόλοιπο corpus: {len(remainder)}")

Αφαιρέθηκαν από το υπόλοιπο corpus: 1186
Υπόλοιπο corpus: 296814


Υπολογίζουμε το μέσο μήκος των προτάσεων στο test set και διαπιστώνουμε ότι είναι σχεδόν ίδιο με αυτό του corpus.

In [5]:
def mean_word_count(frame):
    return pd.Series({
        "greek": frame["greek"].str.split().str.len().mean(),
        "english": frame["english"].str.split().str.len().mean(),
    }).round(1)

pd.DataFrame({"corpus": mean_word_count(corpus), "test": mean_word_count(test_set)})

,corpus,test
greek,11.6,11.7
english,11.7,11.7


## Αποθήκευση

In [6]:
os.makedirs("data", exist_ok=True)

for prefix, frame in [("data/test", test_set), ("work/remainder", remainder)]:
    for extension, column in [("el", "greek"), ("en", "english")]:
        with open(f"{prefix}.{extension}", "w", encoding="utf-8") as f:
            f.write("\n".join(frame[column]) + "\n")